In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

#### 几何参数

In [2]:
d0 = 0.050  # 内管直径 [m]
d1 = 0.130  # 外壳直径 [m]
r0 = d0 / 2  # 内管半径 [m]
r1 = d1 / 2  # 外壳半径 [m]

#### 材料热物性参数

In [3]:
# PCM（石蜡）
# 固相
rho_s = 880.0      # 密度 [kg/m³]
cp_s = 2180.0      # 比热容 [J/(kg·K)]
lambda_s = 0.4     # 导热系数 [W/(m·K)]

# 液相
rho_l = 760.0      # 密度 [kg/m³]
cp_l = 2390.0      # 比热容 [J/(kg·K)]
lambda_l = 0.15    # 导热系数 [W/(m·K)]
mu_l = 0.001       # 粘度 [kg/(m·s)]

# 相变参数
Tpc = 316.15       # 相变温度 [K]
DeltaT = 6.0       # 相变区间 [K]
L = 255000.0       # 相变潜热 [J/kg] (255 kJ/kg)


# 高导热材料（铜）
rho_Cu = 8960.0    # 密度 [kg/m³]
cp_Cu = 385.0      # 比热容 [J/(kg·K)]
lambda_Cu = 400.0  # 导热系数 [W/(m·K)]
mu_Cu = 1e10       # 固体粘度（极大值抑制流动）

# 外壳材料（铝）- 仅用于边界，此处拓扑优化设计域为PCM+铜，铝不参与设计
rho_Al = 2719.0    # 密度 [kg/m³]
cp_Al = 879.0      # 比热容 [J/(kg·K)]
lambda_Al = 202.4  # 导热系数 [W/(m·K)]

#### 物理模型参数

In [4]:
Am = 1e5           # 糊状区常数（论文[40]）
epsilon = 0.001    # 避免分母为零（论文方程4）
alpha = 1e-4       # 热膨胀系数 [1/K]（典型值）
g = 9.81           # 重力加速度 [m/s²]

#### 拓扑优化参数

In [5]:
phi_total = 0.3  # 高导热材料体积比约束（预设，工程常用0.3）
cases_to_train = [1, 2, 3]  # 要训练的case：1-平均温度，2-温度方差，3-多目标

#### 训练超参数 

In [6]:
# 采样点数量
N_train = 10000    # 训练采样点
N_val = 2000       # 验证采样点
N_IC = 3000        # 初始条件采样点
N_BC = 2000        # 边界条件采样点（内管壁+外壳）

# 损失权重（需仔细调整）
lambda_pde = 1.0     # PDE损失权重
lambda_icbc = 10.0   # 初始/边界条件权重
lambda_topo = 100.0  # 拓扑约束权重
lambda_obj = 0.01    # 优化目标权重（初始较小）

# 训练参数
epochs = 8000        # 总训练轮数
init_lr = 1e-3       # 初始学习率
patience = 500       # 学习率调整耐心值

In [7]:
class ResidualBlock(nn.Module):
    """残差块，提高网络表达能力"""
    def __init__(self, dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.activation = nn.Tanh()
        
    def forward(self, x):
        residual = x
        out = self.activation(self.fc1(x))
        out = self.fc2(out)
        return self.activation(out + residual)

In [8]:
class TopoPINN(nn.Module):
    """拓扑优化PINN模型"""
    def __init__(self, input_dim=3, hidden_dim=256, num_layers=6):
        super(TopoPINN, self).__init__()
        
        # 主网络：处理随时间变化的物理场 (ux, uy, p, T)
        layers = [nn.Linear(input_dim, hidden_dim), nn.Tanh()]
        for _ in range(num_layers):
            layers.append(ResidualBlock(hidden_dim))
        layers.append(nn.Linear(hidden_dim, 4))  # 输出: ux, uy, p, T
        self.main_net = nn.Sequential(*layers)
        
        # 拓扑网络：仅处理空间变量 (x, y) -> ρ (设计变量)
        self.topo_net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()  # 输出范围[0,1]
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """权重初始化"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        """
        前向传播
        x: [batch_size, 3] -> (x_norm, y_norm, τ_norm)
        返回: ux, uy, p, T, ρ_design
        """
        # 提取空间坐标 (x, y)
        x_space = x[:, :2]
        
        # 主网络输出
        main_out = self.main_net(x)
        
        # 物理量尺度调整
        ux = main_out[:, 0:1] * 0.01    # 速度尺度: 0.01 m/s
        uy = main_out[:, 1:2] * 0.01
        p = main_out[:, 2:3] * 1000.0   # 压力尺度: 1000 Pa
        T_raw = main_out[:, 3:4]
        
        # 温度约束在合理范围内 [290K, 370K]
        T = 330.0 + 40.0 * torch.tanh(T_raw)  # 330±40K
        
        # 拓扑网络输出（仅依赖于空间坐标）
        rho_design = self.topo_net(x_space)
        
        return ux, uy, p, T, rho_design
    
    def compute_liquid_fraction(self, T):
        """计算液相率 φ(T) - 线性相变模型"""
        T_lower = Tpc - DeltaT/2
        T_upper = Tpc + DeltaT/2
        phi = (T - T_lower) / (T_upper - T_lower)
        return torch.clamp(phi, 0.0, 1.0)

In [9]:
def compute_material_properties(T, rho_design):
    """
    计算混合材料的热物性参数 (PCM + 铜)
    """
    # 1. 计算液相率 φ(T)
    T_lower = Tpc - DeltaT/2
    T_upper = Tpc + DeltaT/2
    phi = (T - T_lower) / (T_upper - T_lower)
    phi = torch.clamp(phi, 0.0, 1.0)
    
    # 2. PCM物性插值
    rho_PCM = rho_s + (rho_l - rho_s) * phi
    lambda_PCM = lambda_s + (lambda_l - lambda_s) * phi
    
    # 3. SIMP材料插值（密度过滤）
    # 使用ρ^3插值以鼓励二值化分布
    rho_design_pow = rho_design**3
    
    # 混合密度和导热系数
    rho_total = rho_design_pow * rho_Cu + (1 - rho_design_pow) * rho_PCM
    lambda_total = rho_design_pow * lambda_Cu + (1 - rho_design_pow) * lambda_PCM
    
    # 4. 糊状区源项 S_t(T)
    S_t = Am * ((1.0 - phi)**2) / (phi**2 + epsilon)
    
    # 5. PCM粘度
    mu_PCM = mu_l + S_t * 1.0  # ξ = 1.0 m²
    
    # 6. 混合粘度（铜为固体，取极大值）
    mu_total = rho_design_pow * mu_Cu + (1 - rho_design_pow) * mu_PCM
    
    # 运动粘度
    nu = mu_total / (rho_total + 1e-10)
    
    # 7. 比热容计算
    # 高斯函数 D(T) 平滑相变潜热
    sigma = DeltaT / 4.0
    D_T = torch.exp(-((T - Tpc)**2) / (sigma**2 + 1e-10)) / (np.sqrt(np.pi) * sigma + 1e-10)
    D_T = torch.clamp(D_T, 0.0, 1e3)
    
    # PCM比热容
    cp_PCM = cp_s + phi * (cp_l - cp_s) + L * D_T
    
    # 混合比热容（质量加权平均）
    mass_Cu = rho_design_pow * rho_Cu
    mass_PCM = (1 - rho_design_pow) * rho_PCM
    mass_total = mass_Cu + mass_PCM + 1e-10
    cp_total = (mass_Cu * cp_Cu + mass_PCM * cp_PCM) / mass_total
    
    # 8. 热扩散率
    a = lambda_total / (rho_total * cp_total + 1e-10)
    
    return {
        'rho': rho_total,
        'lambda': lambda_total,
        'mu': mu_total,
        'nu': nu,
        'cp': cp_total,
        'a': a,
        'S_t': S_t,
        'phi': phi
    }

In [10]:
def sample_points_in_domain(N, t_norm_range=(0.0, 1.0)):
    """
    在计算域内均匀采样点
    返回: points [N, 3] = (x_norm, y_norm, τ_norm), weights [N, 1]
    """
    # 时间采样
    t_norm = np.random.uniform(t_norm_range[0], t_norm_range[1], (N, 1))
    
    # 空间采样（环形域，均匀面积采样）
    # 使用r²均匀分布以确保面积均匀
    r_sq = np.random.uniform(r0**2, r1**2, (N, 1))
    r = np.sqrt(r_sq)
    theta = np.random.uniform(0, 2*np.pi, (N, 1))
    
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    
    # 归一化
    x_norm = x / r1
    y_norm = y / r1
    
    # 面积权重（用于积分）
    weights = r / np.mean(r)
    
    points = np.hstack([x_norm, y_norm, t_norm])
    return (torch.tensor(points, dtype=torch.float32).to(device),
            torch.tensor(weights, dtype=torch.float32).to(device))

def sample_initial_condition(N):
    """初始条件采样点 (τ=0)"""
    points, weights = sample_points_in_domain(N, t_norm_range=(0.0, 0.0))
    return points, weights

def sample_boundary_conditions(N_inner, N_outer):
    """边界条件采样点"""
    # 内管壁 (Dirichlet边界)
    theta_inner = np.random.uniform(0, 2*np.pi, (N_inner, 1))
    x_inner = r0 * np.cos(theta_inner)
    y_inner = r0 * np.sin(theta_inner)
    t_norm_inner = np.random.uniform(0.0, 1.0, (N_inner, 1))
    
    # 外壳 (Neumann边界，绝热)
    theta_outer = np.random.uniform(0, 2*np.pi, (N_outer, 1))
    x_outer = r1 * np.cos(theta_outer)
    y_outer = r1 * np.sin(theta_outer)
    t_norm_outer = np.random.uniform(0.0, 1.0, (N_outer, 1))
    
    # 归一化并组合
    points_inner = np.hstack([x_inner/r1, y_inner/r1, t_norm_inner])
    points_outer = np.hstack([x_outer/r1, y_outer/r1, t_norm_outer])
    
    return (torch.tensor(points_inner, dtype=torch.float32).to(device),
            torch.tensor(points_outer, dtype=torch.float32).to(device))

In [11]:
def compute_pde_loss(model, points, weights):
    """
    计算PDE损失：质量、动量和能量方程
    """
    points.requires_grad_(True)
    ux, uy, p, T, rho_design = model(points)
    
    # 约束温度在合理范围
    T = torch.clamp(T, 290.0, 370.0)
    
    # 计算材料属性
    props = compute_material_properties(T, rho_design)
    rho = props['rho']
    nu = props['nu']
    a = props['a']
    S_t = props['S_t']
    phi = props['phi']
    
    # 1. 质量守恒方程：∇·u = 0
    grad_ux = torch.autograd.grad(ux, points, grad_outputs=torch.ones_like(ux), 
                                  create_graph=True, retain_graph=True)[0]
    grad_uy = torch.autograd.grad(uy, points, grad_outputs=torch.ones_like(uy),
                                  create_graph=True, retain_graph=True)[0]
    
    dudx = grad_ux[:, 0:1]
    dvdy = grad_uy[:, 1:2]
    div_u = dudx + dvdy
    
    # 质量方程损失
    L_mass = torch.mean(div_u**2 * weights)
    
    # 2. 动量守恒方程（论文方程2）
    # x方向动量方程
    convect_x = ux * dudx + uy * grad_ux[:, 1:2]  # (u·∇)ux
    
    grad_p = torch.autograd.grad(p, points, grad_outputs=torch.ones_like(p),
                                 create_graph=True, retain_graph=True)[0]
    dpdx = grad_p[:, 0:1]
    pressure_x = -dpdx / (rho + 1e-10)  # -1/ρ * ∂p/∂x
    
    # 粘性项
    d2udx2 = torch.autograd.grad(dudx, points, grad_outputs=torch.ones_like(dudx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2udy2 = torch.autograd.grad(grad_ux[:, 1:2], points, grad_outputs=torch.ones_like(grad_ux[:, 1:2]),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_x = nu * (d2udx2 + d2udy2)  # ν∇²ux
    
    # 糊状区源项
    source_x = -S_t * ux  # -S_t * ux
    
    # x方向动量残差
    mom_residual_x = convect_x - (pressure_x + viscous_x + source_x)
    
    # y方向动量方程
    convect_y = ux * grad_uy[:, 0:1] + uy * dvdy  # (u·∇)uy
    
    dpdy = grad_p[:, 1:2]
    pressure_y = -dpdy / (rho + 1e-10)  # -1/ρ * ∂p/∂y
    
    # 粘性项
    d2vdx2 = torch.autograd.grad(grad_uy[:, 0:1], points, grad_outputs=torch.ones_like(grad_uy[:, 0:1]),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2vdy2 = torch.autograd.grad(dvdy, points, grad_outputs=torch.ones_like(dvdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    viscous_y = nu * (d2vdx2 + d2vdy2)  # ν∇²uy
    
    # 糊状区源项
    source_y = -S_t * uy  # -S_t * uy
    
    # 浮力项（论文方程3，仅y方向）
    # 注意：浮力仅作用于液态PCM区域
    F_B = rho_l * alpha * g * (T - Tpc) * (1.0 - rho_design) * phi
    
    # y方向动量残差
    mom_residual_y = convect_y - (pressure_y + viscous_y + source_y + F_B)
    
    # 动量方程损失
    L_momentum = torch.mean((mom_residual_x**2 + mom_residual_y**2) * weights)
    
    # 3. 能量方程（论文方程6）
    grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                 create_graph=True, retain_graph=True)[0]
    dTdx = grad_T[:, 0:1]
    dTdy = grad_T[:, 1:2]
    dTdtau_norm = grad_T[:, 2:3]
    
    # 时间导数（注意归一化时间到实际时间）
    dTdtau = dTdtau_norm / 1e5  # τ_norm = τ / 1e5
    
    # 对流项和扩散项
    convect_T = ux * dTdx + uy * dTdy
    d2Tdx2 = torch.autograd.grad(dTdx, points, grad_outputs=torch.ones_like(dTdx),
                                 create_graph=True, retain_graph=True)[0][:, 0:1]
    d2Tdy2 = torch.autograd.grad(dTdy, points, grad_outputs=torch.ones_like(dTdy),
                                 create_graph=True, retain_graph=True)[0][:, 1:2]
    diffusive_T = a * (d2Tdx2 + d2Tdy2)
    
    # 能量方程残差
    energy_residual = dTdtau + convect_T - diffusive_T
    
    # 能量方程损失
    L_energy = torch.mean(energy_residual**2 * weights)
    
    # 总PDE损失
    L_pde = L_mass + L_momentum + L_energy
    
    return L_pde, L_mass, L_momentum, L_energy

In [12]:
def compute_boundary_conditions_loss(model, points_inner, points_outer, heat_storage=True):
    """计算边界条件损失"""
    # 内管壁：Dirichlet边界（恒温）
    _, _, _, T_inner, _ = model(points_inner)
    T_wall = 360.0 if heat_storage else 290.0  # 储热360K，释热290K
    L_dirichlet = torch.mean((T_inner - T_wall)**2)
    
    # 外壳：Neumann边界（绝热，∂T/∂n = 0）
    points_outer.requires_grad_(True)
    _, _, _, T_outer, _ = model(points_outer)
    
    grad_T = torch.autograd.grad(T_outer, points_outer, grad_outputs=torch.ones_like(T_outer),
                                 create_graph=True, retain_graph=True)[0]
    dTdx = grad_T[:, 0:1]
    dTdy = grad_T[:, 1:2]
    
    # 计算法向导数（径向方向）
    x_real = points_outer[:, 0:1] * r1
    y_real = points_outer[:, 1:2] * r1
    r = torch.sqrt(x_real**2 + y_real**2 + 1e-10)
    dTdn = (x_real/r) * dTdx + (y_real/r) * dTdy
    
    L_neumann = torch.mean(dTdn**2)
    
    return L_dirichlet + L_neumann

In [13]:
def compute_initial_conditions_loss(model, points_ic, weights_ic, heat_storage=True):
    """计算初始条件损失"""
    _, _, _, T_ic, _ = model(points_ic)
    T0 = 290.0 if heat_storage else 360.0
    
    # 温度初始条件
    L_temp = torch.mean((T_ic - T0)**2 * weights_ic)
    
    # 速度初始条件（静止）
    ux, uy, _, _, _ = model(points_ic)
    L_vel = torch.mean((ux**2 + uy**2) * weights_ic)
    
    return L_temp + 0.1 * L_vel

def compute_topology_constraints_loss(rho_design, weights):
    """计算拓扑约束损失"""
    # 1. 体积分数约束
    rho_avg = torch.sum(rho_design * weights) / torch.sum(weights + 1e-10)
    vol_residual = torch.relu(rho_avg - phi_total)  # 仅惩罚超过约束
    L_volume = vol_residual**2 * 1000.0
    
    # 2. 取值范围约束 [0, 1]
    L_bounds = torch.mean(torch.relu(-rho_design)**2 + torch.relu(rho_design-1.0)**2)
    
    # 3. SIMP惩罚（鼓励二值化）
    p = 3.0
    L_simp = torch.mean(rho_design**p * (1.0 - rho_design)**p) * 10.0
    
    return L_volume + L_bounds + L_simp, rho_avg

def compute_objective_loss(model, points, weights, case):
    """计算优化目标损失"""
    points.requires_grad_(True)
    _, _, _, T, rho_design = model(points)
    
    # 1. 平均温度
    T_avg = torch.sum(T * weights) / torch.sum(weights + 1e-10)
    L_avg = T_avg**2
    
    # 2. 温度均方差
    T_var = torch.sum(((T - T_avg)**2) * weights) / torch.sum(weights + 1e-10)
    L_var = T_var
    
    # 3. 火积耗散（论文方程16）
    grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                 create_graph=True, retain_graph=True)[0]
    dTdx = grad_T[:, 0:1]
    dTdy = grad_T[:, 1:2]
    
    # 计算导热系数
    props = compute_material_properties(T, rho_design)
    lambda_total = props['lambda']
    
    # φ_g = ∫λ|∇T|² dA
    grad_T_sq = dTdx**2 + dTdy**2
    phi_g = torch.sum(lambda_total * grad_T_sq * weights) / torch.sum(weights + 1e-10)
    L_entransy = phi_g**2
    
    # 根据case选择目标函数
    if case == 1:
        return L_avg, T_avg.item(), T_var.item(), phi_g.item()
    elif case == 2:
        return L_var, T_avg.item(), T_var.item(), phi_g.item()
    else:  # case 3: 多目标
        # 权重设置（可根据需要调整）
        w1, w2, w3 = 1.0, 1.0, 0.1
        L_multi = w1 * L_avg + w2 * L_var + w3 * L_entransy
        return L_multi, T_avg.item(), T_var.item(), phi_g.item()


In [14]:
def train_case(case, epochs=epochs):
    """训练特定case的模型"""
    print(f"\n{'='*60}")
    print(f"开始训练 Case {case}")
    print(f"{'='*60}")
    
    # 创建模型和优化器
    model = TopoPINN(hidden_dim=256, num_layers=6).to(device)
    optimizer = optim.Adam(model.parameters(), lr=init_lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=patience, factor=0.5)
    
    # 采样训练数据
    train_points, train_weights = sample_points_in_domain(N_train)
    ic_points, ic_weights = sample_initial_condition(N_IC)
    bc_inner, bc_outer = sample_boundary_conditions(N_BC//2, N_BC//2)
    
    # 验证数据
    val_points, val_weights = sample_points_in_domain(N_val)
    
    best_loss = float('inf')
    best_epoch = 0
    
    # 训练历史记录
    history = {
        'total_loss': [], 'pde_loss': [], 'icbc_loss': [], 
        'topo_loss': [], 'obj_loss': [], 'rho_avg': []
    }
    
    for epoch in range(epochs):
        model.train()
        
        # 动态调整优化目标权重
        if epoch < 2000:
            obj_weight = 0.0  # 第一阶段：专注物理约束
        elif epoch < 4000:
            obj_weight = lambda_obj * 0.1  # 第二阶段：逐渐引入优化目标
        else:
            obj_weight = lambda_obj  # 第三阶段：完全优化
        
        # 计算各项损失
        # 1. PDE损失
        L_pde, L_mass, L_momentum, L_energy = compute_pde_loss(model, train_points, train_weights)
        
        # 2. 边界条件和初始条件损失
        L_bc = compute_boundary_conditions_loss(model, bc_inner, bc_outer, heat_storage=True)
        L_ic = compute_initial_conditions_loss(model, ic_points, ic_weights, heat_storage=True)
        L_icbc = L_bc + L_ic
        
        # 3. 拓扑约束损失
        _, _, _, _, rho_design = model(train_points)
        L_topo, rho_avg = compute_topology_constraints_loss(rho_design, train_weights)
        
        # 4. 优化目标损失
        if obj_weight > 0:
            L_obj, T_avg_val, T_var_val, phi_g_val = compute_objective_loss(model, val_points, val_weights, case)
        else:
            L_obj = torch.tensor(0.0).to(device)
            T_avg_val, T_var_val, phi_g_val = 0.0, 0.0, 0.0
        
        # 总损失
        L_total = (lambda_pde * L_pde + 
                   lambda_icbc * L_icbc + 
                   lambda_topo * L_topo + 
                   obj_weight * L_obj)
        
        # 反向传播
        optimizer.zero_grad()
        L_total.backward()
        
        # 梯度裁剪（防止梯度爆炸）
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step(L_total.item())
        
        # 记录训练历史
        history['total_loss'].append(L_total.item())
        history['pde_loss'].append(L_pde.item())
        history['icbc_loss'].append(L_icbc.item())
        history['topo_loss'].append(L_topo.item())
        history['obj_loss'].append(L_obj.item() if obj_weight > 0 else 0.0)
        history['rho_avg'].append(rho_avg.item())
        
        # 保存最佳模型
        if L_total.item() < best_loss:
            best_loss = L_total.item()
            best_epoch = epoch
            torch.save(model.state_dict(), f'topo_case{case}_best.pth')
        
        # 打印训练信息
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1}/{epochs}: '
                  f'Total={L_total.item():.2e}, '
                  f'PDE={L_pde.item():.2e} (M={L_mass.item():.2e}, '
                  f'Mom={L_momentum.item():.2e}, E={L_energy.item():.2e}), '
                  f'IC/BC={L_icbc.item():.2e}, '
                  f'Topo={L_topo.item():.2e}, '
                  f'Obj={L_obj.item():.2e if obj_weight > 0 else 0.0:.2e}')
            
            if obj_weight > 0:
                print(f'       T_avg={T_avg_val:.1f}K, T_var={T_var_val:.2e}, '
                      f'Φ_g={phi_g_val:.2e}, ρ_avg={rho_avg.item():.3f}')
            
            # 检查物理量范围
            with torch.no_grad():
                ux_val, uy_val, _, T_val, _ = model(val_points)
                u_mag = torch.sqrt(ux_val**2 + uy_val**2 + 1e-10).mean().item()
                T_mean = T_val.mean().item()
                T_min = T_val.min().item()
                T_max = T_val.max().item()
                
                print(f'       u_mag={u_mag:.2e} m/s, T={T_mean:.1f}K [{T_min:.1f}, {T_max:.1f}]')
        
        # 每500轮重新采样数据（防止过拟合）
        if (epoch + 1) % 500 == 0:
            train_points, train_weights = sample_points_in_domain(N_train)
            ic_points, ic_weights = sample_initial_condition(N_IC)
            bc_inner, bc_outer = sample_boundary_conditions(N_BC//2, N_BC//2)
            val_points, val_weights = sample_points_in_domain(N_val)
    
    # 训练完成
    print(f"\nCase {case} 训练完成！最佳损失: {best_loss:.2e} (Epoch {best_epoch})")
    
    # 加载最佳模型
    model.load_state_dict(torch.load(f'topo_case{case}_best.pth'))
    
    return model, history

In [15]:
def evaluate_model(model, case):
    """评估模型性能"""
    print(f"\n{'='*60}")
    print(f"评估 Case {case} 模型")
    print(f"{'='*60}")
    
    model.eval()
    
    # 1. 计算完全熔化时间
    print("计算完全熔化时间...")
    complete_time = None
    
    for tau in range(0, 100000, 100):  # 0-100000s，步长100s
        tau_norm = tau / 1e5
        points, weights = sample_points_in_domain(2000, (tau_norm, tau_norm))
        
        with torch.no_grad():
            _, _, _, T, _ = model(points)
            phi = model.compute_liquid_fraction(T)
            phi_avg = torch.sum(phi * weights) / torch.sum(weights + 1e-10)
            
            if phi_avg.item() >= 0.98:
                complete_time = tau
                break
    
    # 2. 计算平均储热容量
    avg_heat_storage = 0.0
    if complete_time is not None:
        print(f"完全熔化时间: {complete_time}s")
        print("计算平均储热容量...")
        
        heat_fluxes = []
        time_points = np.linspace(0, min(complete_time, 2000), 20)
        
        for tau in time_points:
            tau_norm = tau / 1e5
            points, weights = sample_points_in_domain(1000, (tau_norm, tau_norm))
            points.requires_grad_(True)
            
            with torch.enable_grad():
                _, _, _, T, rho_design = model(points)
                props = compute_material_properties(T, rho_design)
                lambda_total = props['lambda']
                
                grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                             retain_graph=True)[0]
                dTdx = grad_T[:, 0:1]
                dTdy = grad_T[:, 1:2]
                
                # 计算径向热流
                x_real = points[:, 0:1] * r1
                y_real = points[:, 1:2] * r1
                r = torch.sqrt(x_real**2 + y_real**2 + 1e-10)
                dTdr = (x_real/r) * dTdx + (y_real/r) * dTdy
                
                # 热流密度 q = -λ dT/dr
                q = -lambda_total * dTdr
                q_avg = torch.sum(q * weights) / torch.sum(weights + 1e-10)
                heat_fluxes.append(q_avg.item())
        
        avg_heat_storage = np.mean(heat_fluxes)
        print(f"平均储热容量: {avg_heat_storage:.2f} W")
    
    # 3. 计算场协同角
    print("计算场协同角...")
    synergy_angles = []
    
    for tau in [200, 400, 600, 800]:
        tau_norm = tau / 1e5
        points, _ = sample_points_in_domain(1000, (tau_norm, tau_norm))
        points.requires_grad_(True)
        
        with torch.enable_grad():
            ux, uy, _, T, _ = model(points)
            grad_T = torch.autograd.grad(T, points, grad_outputs=torch.ones_like(T),
                                         retain_graph=True)[0]
            
            # 计算场协同角
            u_dot_gradT = ux * grad_T[:, 0:1] + uy * grad_T[:, 1:2]
            norm_u = torch.sqrt(ux**2 + uy**2 + 1e-10)
            norm_gradT = torch.sqrt(grad_T[:, 0:1]**2 + grad_T[:, 1:2]**2 + 1e-10)
            
            cos_theta = u_dot_gradT / (norm_u * norm_gradT + 1e-10)
            cos_theta = torch.clamp(cos_theta, -1.0, 1.0)
            theta = torch.acos(cos_theta) * 180.0 / np.pi
            
            synergy_angles.append(theta.mean().item())
    
    avg_synergy = np.mean(synergy_angles)
    print(f"平均场协同角: {avg_synergy:.1f}°")
    
    # 4. 计算优化目标值
    print("计算优化目标值...")
    points, weights = sample_points_in_domain(N_val)
    with torch.no_grad():
        _, T_avg, T_var, phi_g = compute_objective_loss(model, points, weights, case)
    
    print(f"平均温度: {T_avg:.2f} K")
    print(f"温度均方差: {T_var:.4f}")
    print(f"火积耗散: {phi_g:.4e}")
    
    return {
        'case': case,
        'complete_time': complete_time,
        'avg_heat_storage': avg_heat_storage,
        'avg_synergy': avg_synergy,
        'T_avg': T_avg,
        'T_var': T_var,
        'phi_g': phi_g
    }

In [16]:
def visualize_results(model, case, results, save_dir="./results"):
    """可视化结果"""
    os.makedirs(save_dir, exist_ok=True)
    case_dir = os.path.join(save_dir, f"case{case}")
    os.makedirs(case_dir, exist_ok=True)
    
    # 生成网格数据
    nr, ntheta = 100, 100
    r = np.linspace(r0, r1, nr)
    theta = np.linspace(0, 2*np.pi, ntheta)
    R, Theta = np.meshgrid(r, theta)
    X = R * np.cos(Theta)
    Y = R * np.sin(Theta)
    
    # 选择时间点
    time_points = [100, 300, 500, 751, 1000]
    
    for tau in time_points:
        tau_norm = tau / 1e5
        
        # 准备输入数据
        x_flat = X.reshape(-1, 1) / r1
        y_flat = Y.reshape(-1, 1) / r1
        t_flat = np.full_like(x_flat, tau_norm)
        points = np.hstack([x_flat, y_flat, t_flat])
        
        points_tensor = torch.tensor(points, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            ux, uy, p, T, rho_design = model(points_tensor)
            phi = model.compute_liquid_fraction(T)
        
        # 转换为网格格式
        T_grid = T.cpu().numpy().reshape(ntheta, nr)
        phi_grid = phi.cpu().numpy().reshape(ntheta, nr)
        rho_grid = rho_design.cpu().numpy().reshape(ntheta, nr)
        ux_grid = ux.cpu().numpy().reshape(ntheta, nr)
        uy_grid = uy.cpu().numpy().reshape(ntheta, nr)
        
        # 计算速度大小
        u_mag = np.sqrt(ux_grid**2 + uy_grid**2)
        
        # 创建图形
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        fig.suptitle(f'Case {case} - Time = {tau}s', fontsize=16)
        
        # 温度分布
        im1 = axes[0, 0].contourf(X, Y, T_grid, levels=50, cmap='jet', vmin=290, vmax=360)
        axes[0, 0].set_title('Temperature [K]')
        axes[0, 0].set_aspect('equal')
        plt.colorbar(im1, ax=axes[0, 0])
        
        # 液相率
        im2 = axes[0, 1].contourf(X, Y, phi_grid, levels=50, cmap='viridis', vmin=0, vmax=1)
        axes[0, 1].set_title('Liquid Fraction')
        axes[0, 1].set_aspect('equal')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # 拓扑结构
        im3 = axes[0, 2].contourf(X, Y, rho_grid, levels=50, cmap='binary', vmin=0, vmax=1)
        axes[0, 2].set_title('Topology (ρ)')
        axes[0, 2].set_aspect('equal')
        plt.colorbar(im3, ax=axes[0, 2])
        
        # 速度大小
        im4 = axes[1, 0].contourf(X, Y, u_mag, levels=50, cmap='plasma', norm=plt.LogNorm(vmin=1e-6, vmax=1e-2))
        axes[1, 0].set_title('Velocity Magnitude [m/s]')
        axes[1, 0].set_aspect('equal')
        plt.colorbar(im4, ax=axes[1, 0])
        
        # 速度矢量（抽样显示）
        skip = 5
        axes[1, 1].quiver(X[::skip, ::skip], Y[::skip, ::skip], 
                          ux_grid[::skip, ::skip], uy_grid[::skip, ::skip],
                          scale=0.1, color='red')
        axes[1, 1].set_title('Velocity Vectors')
        axes[1, 1].set_aspect('equal')
        axes[1, 1].set_xlim([-r1, r1])
        axes[1, 1].set_ylim([-r1, r1])
        
        # 压力分布
        p_grid = p.cpu().numpy().reshape(ntheta, nr)
        im6 = axes[1, 2].contourf(X, Y, p_grid, levels=50, cmap='coolwarm')
        axes[1, 2].set_title('Pressure [Pa]')
        axes[1, 2].set_aspect('equal')
        plt.colorbar(im6, ax=axes[1, 2])
        
        plt.tight_layout()
        plt.savefig(os.path.join(case_dir, f'tau_{tau}s.png'), dpi=300, bbox_inches='tight')
        plt.close()
    
    # 保存结果摘要
    with open(os.path.join(case_dir, 'results_summary.txt'), 'w') as f:
        f.write(f"Case {case} Results Summary\n")
        f.write("="*40 + "\n")
        f.write(f"Complete melting time: {results['complete_time']} s\n")
        f.write(f"Average heat storage capacity: {results['avg_heat_storage']:.2f} W\n")
        f.write(f"Average synergy angle: {results['avg_synergy']:.1f}°\n")
        f.write(f"Average temperature: {results['T_avg']:.2f} K\n")
        f.write(f"Temperature variance: {results['T_var']:.4f}\n")
        f.write(f"Entransy dissipation: {results['phi_g']:.4e}\n")
    
    print(f"Case {case} 结果已保存到 {case_dir}")

In [17]:
if __name__ == "__main__":
    print("拓扑优化相变储热系统PINN求解器")
    print(f"开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"训练案例: {cases_to_train}")
    
    # 存储所有结果
    all_results = []
    
    for case in cases_to_train:
        print(f"\n{'#'*80}")
        print(f"处理 Case {case}")
        print(f"{'#'*80}")
        
        # 训练模型
        model, history = train_case(case, epochs=epochs)
        
        # 评估模型
        results = evaluate_model(model, case)
        all_results.append(results)
        
        # 可视化结果
        visualize_results(model, case, results, save_dir="./topo_results")
        
        # 清理内存
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # 打印结果对比
    print(f"\n{'='*80}")
    print("所有Case结果对比")
    print(f"{'='*80}")
    
    print("\nCase | 熔化时间(s) | 储热容量(W) | 场协同角(°) | 平均温度(K) | 温度方差 | 火积耗散")
    print("-"*80)
    
    for result in all_results:
        print(f"{result['case']:4d} | "
              f"{result['complete_time'] if result['complete_time'] else 'N/A':10} | "
              f"{result['avg_heat_storage']:11.2f} | "
              f"{result['avg_synergy']:11.1f} | "
              f"{result['T_avg']:11.2f} | "
              f"{result['T_var']:9.2e} | "
              f"{result['phi_g']:9.2e}")
    
    # 创建对比图表
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 提取数据
    cases = [r['case'] for r in all_results]
    melting_times = [r['complete_time'] if r['complete_time'] else 0 for r in all_results]
    heat_capacities = [r['avg_heat_storage'] for r in all_results]
    synergy_angles = [r['avg_synergy'] for r in all_results]
    avg_temps = [r['T_avg'] for r in all_results]
    temp_vars = [r['T_var'] for r in all_results]
    entransy_diss = [r['phi_g'] for r in all_results]
    
    # 绘制对比图
    axes[0, 0].bar(cases, melting_times, color=['blue', 'orange', 'green'])
    axes[0, 0].set_title('Complete Melting Time')
    axes[0, 0].set_xlabel('Case')
    axes[0, 0].set_ylabel('Time (s)')
    
    axes[0, 1].bar(cases, heat_capacities, color=['blue', 'orange', 'green'])
    axes[0, 1].set_title('Average Heat Storage Capacity')
    axes[0, 1].set_xlabel('Case')
    axes[0, 1].set_ylabel('Power (W)')
    
    axes[0, 2].bar(cases, synergy_angles, color=['blue', 'orange', 'green'])
    axes[0, 2].set_title('Average Synergy Angle')
    axes[0, 2].set_xlabel('Case')
    axes[0, 2].set_ylabel('Angle (°)')
    
    axes[1, 0].bar(cases, avg_temps, color=['blue', 'orange', 'green'])
    axes[1, 0].set_title('Average Temperature')
    axes[1, 0].set_xlabel('Case')
    axes[1, 0].set_ylabel('Temperature (K)')
    
    axes[1, 1].bar(cases, temp_vars, color=['blue', 'orange', 'green'])
    axes[1, 1].set_title('Temperature Variance')
    axes[1, 1].set_xlabel('Case')
    axes[1, 1].set_ylabel('Variance')
    axes[1, 1].set_yscale('log')
    
    axes[1, 2].bar(cases, entransy_diss, color=['blue', 'orange', 'green'])
    axes[1, 2].set_title('Entransy Dissipation')
    axes[1, 2].set_xlabel('Case')
    axes[1, 2].set_ylabel('Φ_g')
    axes[1, 2].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig("./topo_results/comparison_summary.png", dpi=300, bbox_inches='tight')
    
    print(f"\n程序执行完成！结果已保存到 ./topo_results/")
    print(f"结束时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

拓扑优化相变储热系统PINN求解器
开始时间: 2026-01-02 11:56:35
训练案例: [1, 2, 3]

################################################################################
处理 Case 1
################################################################################

开始训练 Case 1


ValueError: Invalid format specifier '.2e if obj_weight > 0 else 0.0:.2e' for object of type 'float'